# Rising Waters Flood Prediction System

This notebook implements Epic 1, Epic 2, and Epic 3 for the Rising Waters Flood Prediction System. It loads the dataset, explores and visualizes the data, prepares features, splits the dataset, scales the input features, and saves the fitted scaler.

# Epic 1 - Data Collection

In this section, we import pandas and load the flood dataset from the Excel file using `pd.read_excel()`.

In [ ]:
# Import tools for checking and installing the Excel reader dependency
import importlib.util
import subprocess
import sys

# Install openpyxl if it is missing from the active notebook environment
if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])

# Import pandas for reading and working with tabular data
import pandas as pd

# Load the dataset from the data folder
df = pd.read_excel("../data/flood_dataset.xlsx", engine="openpyxl")

# Print a success message after loading the dataset
print("Dataset loaded successfully.")

# Epic 2 - Data Visualization and Analysis

This section focuses on understanding the dataset through exploration, descriptive statistics, univariate analysis, and multivariate analysis.

## 1. Import Libraries

These libraries are used for numerical operations, data analysis, and visualizations.

In [ ]:
# Import libraries required for analysis and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Dataset Exploration

Dataset exploration helps us understand the structure, size, column names, data types, summary statistics, and missing values in the dataset.

In [ ]:
# Display the first five rows of the dataset
df.head()

In [ ]:
# Display the number of rows and columns
df.shape

In [ ]:
# Display all column names
df.columns

In [ ]:
# Display dataset information, including non-null values and data types
df.info()

In [ ]:
# Display descriptive statistics for numerical columns
df.describe()

In [ ]:
# Display the data type of each column
df.dtypes

In [ ]:
# Display missing value counts for each column
df.isnull().sum()

## 3. Descriptive Statistical Analysis

Descriptive statistical analysis summarizes the dataset using statistics, data types, unique value counts, missing value counts, and a simple dataset summary.

In [ ]:
# Display descriptive statistics
display(df.describe())

# Display data types
display(df.dtypes)

# Display number of unique values in each column
display(df.nunique())

# Display missing values summary
missing_values_summary = df.isnull().sum()
display(missing_values_summary)

# Print a beginner-friendly dataset summary
total_rows, total_columns = df.shape
numerical_columns = df.select_dtypes(include=[np.number]).columns.tolist()
target_column = "flood"

print(f"Total rows: {total_rows}")
print(f"Total columns: {total_columns}")
print(f"Numerical columns: {numerical_columns}")
print(f"Target column: {target_column}")
print(f"Missing values exist: {missing_values_summary.sum() > 0}")

## 4. Univariate Analysis

Univariate analysis studies one feature at a time. Histograms with KDE help show the distribution of each numerical feature, and boxplots help show spread and possible outliers.

The target column `flood` is excluded from this feature-level analysis.

In [ ]:
# Automatically detect numerical columns
numerical_columns = df.select_dtypes(include=[np.number]).columns.tolist()

# Exclude the target column from numerical feature analysis
numerical_features = [column for column in numerical_columns if column.lower() != "flood"]

# Display the selected numerical features
numerical_features

In [ ]:
# Create histogram with KDE for every numerical feature
for feature in numerical_features:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=df, x=feature, kde=True)
    plt.title(f"Histogram with KDE of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

In [ ]:
# Create boxplot for every numerical feature
for feature in numerical_features:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x=feature)
    plt.title(f"Boxplot of {feature}")
    plt.xlabel(feature)
    plt.tight_layout()
    plt.show()

## 5. Multivariate Analysis

Multivariate analysis studies relationships between multiple variables. It helps identify features that may be strongly related to the target column `flood`.

In [ ]:
# Select only numerical columns for correlation analysis
numerical_df = df.select_dtypes(include=[np.number])

# Compute the correlation matrix
correlation_matrix = numerical_df.corr()

# Display the correlation matrix
correlation_matrix

In [ ]:
# Create a correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Print features most correlated with the target column
if "flood" in correlation_matrix.columns:
    target_correlations = correlation_matrix["flood"].drop("flood")
    target_correlations = target_correlations.reindex(
        target_correlations.abs().sort_values(ascending=False).index
    )
    
    print("Features most correlated with 'flood':")
    print(target_correlations)
else:
    print("The target column 'flood' was not found in the numerical columns.")

In [ ]:
# Create scatter plots for selected features against flood when columns exist
scatter_features = ["ANNUAL", "Jun-Sep", "Humidity"]

for feature in scatter_features:
    if feature in df.columns and "flood" in df.columns:
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=df, x=feature, y="flood")
        plt.title(f"{feature} vs flood")
        plt.xlabel(feature)
        plt.ylabel("flood")
        plt.tight_layout()
        plt.show()
    else:
        print(f"Skipping {feature} vs flood because one or both columns are missing.")

# Epic 3 - Data Preprocessing

Data preprocessing prepares the dataset for machine learning by checking missing values, handling outliers, encoding categorical features, splitting the data, and scaling numerical input features.

## 1. Missing Value Analysis

Missing value analysis checks whether any columns contain empty or null values. This is important because many machine learning algorithms cannot handle missing values directly.

In [ ]:
# Create a working copy for preprocessing
dataset = df.copy()

# Count missing values in each column
missing_values_count = dataset.isnull().sum()
display(missing_values_count)

# Check whether each column contains any missing values
missing_values_any = dataset.isnull().any()
display(missing_values_any)

# Print whether missing values exist in the dataset
if missing_values_count.sum() > 0:
    print("Missing values exist in the dataset.")
else:
    print("No missing values exist in the dataset.")

## 2. Outlier Handling

Outliers are detected using the Interquartile Range (IQR) method. Instead of deleting rows, extreme values are capped using `clip()` so the dataset size remains unchanged.

In [ ]:
# Detect numerical columns again from the preprocessing dataset
numerical_columns = dataset.select_dtypes(include=[np.number]).columns.tolist()

# Exclude the target column from outlier handling
outlier_features = [column for column in numerical_columns if column.lower() != "flood"]

# Apply IQR-based capping for each numerical feature
for feature in outlier_features:
    Q1 = dataset[feature].quantile(0.25)
    Q3 = dataset[feature].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    below_lower_bound = dataset[feature] < lower_bound
    above_upper_bound = dataset[feature] > upper_bound
    capped_values_count = below_lower_bound.sum() + above_upper_bound.sum()
    
    dataset[feature] = dataset[feature].clip(lower=lower_bound, upper=upper_bound)
    
    print(f"{feature}: {capped_values_count} value(s) capped")

## 3. Feature Preparation

Categorical columns must be converted into numbers before machine learning. If the dataset contains only numerical features, label encoding is not required.

In [ ]:
# Import LabelEncoder for categorical feature conversion
from sklearn.preprocessing import LabelEncoder

# Detect categorical columns
categorical_columns = dataset.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# Apply LabelEncoder only if categorical columns exist
if categorical_columns:
    print(f"Categorical columns detected: {categorical_columns}")
    label_encoder = LabelEncoder()
    
    for column in categorical_columns:
        dataset[column] = label_encoder.fit_transform(dataset[column].astype(str))
        print(f"Label encoded column: {column}")
else:
    print("The dataset contains only numerical features. Therefore, label encoding is not required.")

## 4. Create Independent and Dependent Variables

`X` contains the independent variables, which are the input features. `y` contains the dependent variable, which is the target column `flood`.

In [ ]:
# Create independent variables by dropping the target column
X = dataset.drop("flood", axis=1)

# Create dependent variable from the target column
y = dataset["flood"]

# Print the shapes of X and y
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

## 5. Train-Test Split

The dataset is split into training and testing sets. The training set is used for learning later, and the testing set is used to evaluate future model performance.

In [ ]:
# Import train_test_split for splitting the dataset
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Print the shapes of the split datasets
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

## 6. Feature Scaling

Feature scaling standardizes input features so they have a similar scale. `StandardScaler` is fitted on the training data and then used to transform both the training and testing data.

In [ ]:
# Import StandardScaler for feature scaling and joblib for saving the scaler
from sklearn.preprocessing import StandardScaler
import joblib

# Create the scaler object
scaler = StandardScaler()

# Fit the scaler on training data and transform training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform testing data using the same fitted scaler
X_test_scaled = scaler.transform(X_test)

# Save the fitted scaler for future use
joblib.dump(scaler, "../models/transform.save")

print("Feature scaling completed successfully.")
print("Scaler saved successfully at ../models/transform.save")